In [1]:
!git clone https://github.com/HenriqueSchmitz/mario-the-explorer

Cloning into 'mario-the-explorer'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 118 (delta 41), reused 102 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 469.49 KiB | 4.99 MiB/s, done.
Resolving deltas: 100% (41/41), done.


In [2]:
!sh ./mario-the-explorer/setup.sh

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.5 MB/s eta 0:00:00
Importing SuperMarioWorld-Snes-v0
Imported 1 games


In [3]:
from typing import Optional

import numpy as np

from mario_the_explorer import SuperMarioWorldEmulator, RewardModel, ScreenOverlay, Tile, get_file_logger

In [4]:
class NoRewardModel(RewardModel):

    def get_reward(self,
                   action: list[int],
                   observation: list[list[Tile]],
                   terminated: bool,
                   truncated: bool,
                   info: dict) -> float:
        return 0.0

In [5]:
class NoOverlay(ScreenOverlay):

    def apply(self, original_frame: np.ndarray, observation: Optional[list[list[Tile]]]):
        return original_frame

In [6]:
WALK_RIGHT_ACTION = [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]

In [7]:
RUN_NAME = "test_run"
LEVEL = "DonutPlains1"
MAX_STEPS = 1500
LOG_LEVEL = "INFO"

In [8]:
import traceback
from gymnasium.wrappers import RecordVideo


logger = get_file_logger(RUN_NAME, LOG_LEVEL)
env = SuperMarioWorldEmulator(level = LEVEL, # str
                              render_mode = "rgb_array", # str
                              reward_model = NoRewardModel(), # RewardModel
                              screen_overlay = NoOverlay(), # ScreenOverlay, optional
                              max_episode_length = MAX_STEPS, # int, optional
                              render_debug = True, # bool, optional
                              render_grid = True, # bool, optional
                              logger = logger) # Logger, optional
try:
    env = RecordVideo(env, video_folder="./", name_prefix=RUN_NAME, episode_trigger=lambda x: True)
    obs = env.reset()
    done = False
    while not done:
        env.render()
        obs, reward, terminated, truncated, info = env.step(WALK_RIGHT_ACTION)
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(traceback.format_exc())
    raise e
finally:
    env.close()

2026-04-21 21:14:56 [INFO] Session log for run test_run with level [INFO] initialized at: test_run_20260421_211456.log
/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
2026-04-21 21:15:03 [INFO] Terminated: True
2026-04-21 21:15:03 [INFO] Truncated: False
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
